# Question 1: Trace values and notebook state

### My prediction before running the code
1. First printed line: [1200, 800, 0]
2. Second printed line: 600 3
3. Third printed line: 20%

In [1]:
incomes = [1000, 800, 0]
saved_incomes = incomes
first_share = (incomes[0] - 800) / incomes[0]
incomes[0] = 1200
saved_incomes.append(600)

print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0, 600]
600 4
20%


- incomes[0] changes from 1000 to 1200. saved_incomes refers to the same list as incomes, so appending 600 changes that list too. It now has four values. The last value is 600, and the list length is 4.
- first_share was calculated before the income changed therefore changing an income does not update first_share: (1000 - 800) / 1000 = 0.2, displayed as 20%.

In [2]:

incomes = [1000, 800, 0]
saved_incomes = incomes.copy()
incomes[0] = 1200
first_share = (incomes[0] - 800) / incomes[0]
saved_incomes.append(600)

print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0]
600 3
33%



- copy() created an independent list so incomes is separated from saved_incomes.
- incomes[0] is changed from 1000 to 1200 before first_share calculation so result of last print changed also: (1200 - 800) / 1200


# Question 2: Implement the policy as functions

In [3]:
# (a) saving_share(income, consumption)

def saving_share(income, consumption):
    if income < 0 or consumption < 0:
        raise ValueError("Income and consumption must be non-negative")
    if income == 0:
        return None
    return (income - consumption) / income

- `income` and `consumption` are parameters.
- Supplied values are arguments.
- Checks precede division to reject invalid amounts and prevent division by zero.
- `return` passes the result back to the calling code so it can use the value later.



In [4]:
# (b) transfer(income, consumption, size)

def transfer(income, consumption, size):
    if income < 0 or consumption < 0 or size <= 0:
        raise ValueError("Income and consumption most be non-negative, size must be positive")
    if income / size > 600:
        return 0
    return min(100 * size, max(consumption - income, 0))

- Validates inputs before eligibility.
- Size must be positive before division and negative amounts are rejected.

In [5]:
# (c) post_policy_share(income, consumption, size)

def post_policy_share(income, consumption, size):
    support = transfer(income, consumption, size)
    income_after_support = income + support
    return saving_share(income_after_support, consumption)

- This function uses returns from functions defined before. Reusing these functions avoids duplicating their calculations.


# Question 3: Build and interpret a checked report

In [6]:
# (a) Prepare inputs

household_ids = ["A01", "A02", "A03", "A04",
                 "A05", "A06", "A07", "A08"]
income_text = ["1200", "0", "1500", "700",
               "2400", "800", "0", "1800"]
consumptions = [1250, 150, 1600, 850, 2600, 950, 0, 1400]
household_sizes = [2, 1, 3, 1, 4, 2, 2, 3]

if not (len(household_ids) == len(income_text)
        == len(consumptions) == len(household_sizes)):
    raise ValueError("The input lists must have equal lengths")

incomes = []
for amount in income_text:
    incomes.append(int(amount))

In [7]:
# (b) Calculate household results

transfers = []
savings_before = []
savings_after = []
shares_before = []
shares_after = []

print(
    "ID | Income | Consumption | Size | Transfer | "
    "Saving before | Saving after | Share before | Share after"
) #This prints labels by telling what each value on a report line represents. The | characters are visual separators.

for i in range(len(household_ids)):
    household_id = household_ids[i]
    income = incomes[i]
    consumption = consumptions[i]
    size = household_sizes[i]

    support = transfer(income, consumption, size)
    income_after = income + support
    saving_before = income - consumption
    saving_after = income_after - consumption
    share_before = saving_share(income, consumption)
    share_after = post_policy_share(income, consumption, size)

    transfers.append(support)
    savings_before.append(saving_before)
    savings_after.append(saving_after)
    shares_before.append(share_before)
    shares_after.append(share_after)

    if share_before is None:
        share_before_display = "undefined"
    else:
        share_before_display = f"{share_before:.2%}"

    if share_after is None:
        share_after_display = "undefined"
    else:
        share_after_display = f"{share_after:.2%}"

    print(
        f"{household_id} | {income:.2f} | {consumption:.2f} | {size} | "
        f"{support:.2f} | {saving_before:.2f} | {saving_after:.2f} | "
        f"{share_before_display} | {share_after_display}"
    )

ID | Income | Consumption | Size | Transfer | Saving before | Saving after | Share before | Share after
A01 | 1200.00 | 1250.00 | 2 | 50.00 | -50.00 | 0.00 | -4.17% | 0.00%
A02 | 0.00 | 150.00 | 1 | 100.00 | -150.00 | -50.00 | undefined | -50.00%
A03 | 1500.00 | 1600.00 | 3 | 100.00 | -100.00 | 0.00 | -6.67% | 0.00%
A04 | 700.00 | 850.00 | 1 | 0.00 | -150.00 | -150.00 | -21.43% | -21.43%
A05 | 2400.00 | 2600.00 | 4 | 200.00 | -200.00 | 0.00 | -8.33% | 0.00%
A06 | 800.00 | 950.00 | 2 | 150.00 | -150.00 | 0.00 | -18.75% | 0.00%
A07 | 0.00 | 0.00 | 2 | 0.00 | 0.00 | 0.00 | undefined | undefined
A08 | 1800.00 | 1400.00 | 3 | 0.00 | 400.00 | 400.00 | 22.22% | 22.22%


In [8]:
# (c) Compare totals and averages

total_transfer = sum(transfers)
total_income_before = sum(incomes)
total_saving_before = sum(savings_before)
total_income_after = total_income_before + total_transfer
total_saving_after = sum(savings_after)

positive_transfer_count = 0
negative_saving_before_count = 0
negative_saving_after_count = 0

before_share_total = 0
before_share_count = 0
after_share_total = 0
after_share_count = 0
matched_after_share_total = 0
matched_count = 0

for i in range(len(household_ids)):
    if transfers[i] > 0:
        positive_transfer_count += 1

    if savings_before[i] < 0:
        negative_saving_before_count += 1

    if savings_after[i] < 0:
        negative_saving_after_count += 1

    if shares_before[i] is not None:
        before_share_total += shares_before[i]
        before_share_count += 1

        # These are the same households used in the before-support mean.
        matched_after_share_total += shares_after[i]
        matched_count += 1

    if shares_after[i] is not None:
        after_share_total += shares_after[i]
        after_share_count += 1

pooled_share_before = total_saving_before / total_income_before
pooled_share_after = total_saving_after / total_income_after
mean_share_before = before_share_total / before_share_count
mean_share_after = after_share_total / after_share_count
matched_mean_share_after = matched_after_share_total / matched_count

print(f"Total transfer: €{total_transfer:.2f}")
print(f"Households receiving a positive transfer: {positive_transfer_count}")
print(f"Households with negative saving before: {negative_saving_before_count}")
print(f"Households with negative saving after: {negative_saving_after_count}")
print(f"Pooled saving share before: {pooled_share_before:.2%}")
print(f"Pooled saving share after: {pooled_share_after:.2%}")
print(f"Mean defined household share before: {mean_share_before:.2%} "
      f"(n={before_share_count})")
print(f"Mean defined household share after: {mean_share_after:.2%} "
      f"(n={after_share_count})")
print(f"Mean after for households with defined before share: "
      f"{matched_mean_share_after:.2%} (n={matched_count})")

Total transfer: €600.00
Households receiving a positive transfer: 5
Households with negative saving before: 6
Households with negative saving after: 2
Pooled saving share before: -4.76%
Pooled saving share after: 2.22%
Mean defined household share before: -6.19% (n=6)
Mean defined household share after: -7.03% (n=7)
Mean after for households with defined before share: 0.13% (n=6)


The pooled saving share rises from −4.76% to 2.22%, while the mean across all defined household shares falls from −6.19% to −7.03%. These summaries differ because the pooled share weights households by income, while the household mean gives each included household equal weight. The after-support mean includes A02, whose share was undefined before support but becomes −50% after support, so the two means use different groups. Comparing the same six households, the mean rises from −6.19% before support to 0.13% after support. Means over different groups can make the change look misleading.

In [9]:
# (d) Use an inline function

defined_before_shares = []

for share in shares_before:
    if share is not None:
        defined_before_shares.append(share)

shares_nearest_to_target = sorted(
    defined_before_shares,
    key=lambda share: abs(share - 0.10)
)

print(shares_nearest_to_target)

[0.2222222222222222, -0.041666666666666664, -0.06666666666666667, -0.08333333333333333, -0.1875, -0.21428571428571427]


The `lambda` receives one share at a time and returns its absolute distance from 0.10. `sorted` uses those distances to decide the order, but returns the original share values. The new list excludes `None`; `shares_before` stays unchanged.

In [10]:
# (e) Test a counterfactual

original_incomes = incomes.copy()
counterfactual_incomes = incomes.copy()
counterfactual_incomes[0] = 1202 #income of A01 raises by 2 (from 1200 to 1202)

consumption = consumptions[0]
size = household_sizes[0]

original_transfer = transfer(incomes[0], consumption, size)
counterfactual_transfer = transfer(
    counterfactual_incomes[0], consumption, size
)

original_income_after = incomes[0] + original_transfer
counterfactual_income_after = (
    counterfactual_incomes[0] + counterfactual_transfer
)

original_saving_after = original_income_after - consumption
counterfactual_saving_after = counterfactual_income_after - consumption

print("Quantity | Original | Counterfactual | Change")
print(
    f"Income | {incomes[0]:.2f} | "
    f"{counterfactual_incomes[0]:.2f} | "
    f"{counterfactual_incomes[0] - incomes[0]:.2f}"
)
print(
    f"Transfer | {original_transfer:.2f} | "
    f"{counterfactual_transfer:.2f} | "
    f"{counterfactual_transfer - original_transfer:.2f}"
)
print(
    f"Income after support | {original_income_after:.2f} | "
    f"{counterfactual_income_after:.2f} | "
    f"{counterfactual_income_after - original_income_after:.2f}"
)
print(
    f"Saving after support | {original_saving_after:.2f} | "
    f"{counterfactual_saving_after:.2f} | "
    f"{counterfactual_saving_after - original_saving_after:.2f}"
)
print("Original income list unchanged:", incomes == original_incomes)

Quantity | Original | Counterfactual | Change
Income | 1200.00 | 1202.00 | 2.00
Transfer | 50.00 | 0.00 | -50.00
Income after support | 1250.00 | 1202.00 | -48.00
Saving after support | 0.00 | -48.00 | -48.00
Original income list unchanged: True


Higher original income does not necessarily mean higher income after support. A01’s increase from €1,200 to €1,202 raises income per person from €600 to €601, crossing the eligibility cutoff. The household loses its €50 transfer, so income after support falls by €48. The policy’s threshold causes this result.

Saving after support is an amount in euros; the saving share is that amount divided by income after support. For A01, saving after support changes from €0 to −€48. The corresponding share changes from 0% to about −3.99%. These are related measures, but a change in the euro amount is not the same as a change in the share.

# Question 4: Audit an agent-written function 

In [11]:
# Agent proposed this function:

def proposed_mean(incomes, consumptions):
    total = 0
    count = 0
    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None and share != 0:
            total = 0
            total = total + share
            count = count + 1
    return total / count

(a)
1. **Unequal list lengths:** `zip(incomes, consumptions)` stops at the *shorter list*, silently omitting extra households. The function should raise `ValueError` when the lists have different lengths.

2. **Zero shares are excluded:** The condition `share != 0` skips valid zero shares. The contract says to include *all defined shares*, including zero and negative values.

3. **The running total is reset:** `total = 0` is inside the loop, so each included share erases the total accumulated from earlier households. As a result, the function does not calculate the correct mean.

4. **No defined shares cause an error:** If the inputs are empty or every share is `None`, `count` stays zero. Then `total / count` raises `ZeroDivisionError`, but the function should return `None`.

In [12]:
# (b)

def mean_defined_share(incomes, consumptions):
    if len(incomes) != len(consumptions):
        raise ValueError("Input lists must have equal lengths")

    total = 0
    count = 0

    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None:
            total += share
            count += 1

    if count == 0:
        return None

    return total / count

The length check makes sure `zip` won’t silently omit an item. The `is not None` condition includes zero and negative shares. If there are no defined shares, the function returns `None` instead of dividing by zero. It reads the input lists without changing them.

(c)

| Inputs: `incomes`, `consumptions` | Correct outcome | Proposal result | Defect exposed |
|---|---|---|---|
| `[100, 200]`, `[50]` | Raise `ValueError` because lengths differ | Returns `0.5`; `zip` processes only the first pair | Unequal lengths are not checked |
| `[100, 100]`, `[100, 0]` | Mean of shares `0` and `1` = `0.5` | Returns `1.0`; it skips the zero share | Valid zero shares are excluded |
| `[100, 100]`, `[0, 50]` | Mean of shares `1` and `0.5` = `0.75` | Returns `0.25`; it resets the total before adding the second share | Running total is reset inside the loop |
| `[]`, `[]` | Return `None` because there are no defined shares | Raises `ZeroDivisionError` when dividing by zero | No-defined-shares case is not handled |

The proposed function fails when there are no defined shares:

```text
proposed_mean([], [])
ZeroDivisionError: division by zero
```

The corrected function raises the required error when the input lists have different lengths:

```text
mean_defined_share([100, 200], [50])
ValueError: Input lists must have equal lengths
```

# Question 5: Supply evidence and a reproducible hand-in

In [13]:
# (a)

import math

assert math.isclose(saving_share(1000, 800), 0.2)
assert math.isclose(saving_share(800, 950), -0.1875)
assert saving_share(0, 100) is None
assert math.isclose(transfer(1200, 1250, 2), 50)
assert math.isclose(transfer(1200, 1600, 2), 200)
assert math.isclose(transfer(1200, 1000, 2), 0)
assert mean_defined_share([0, 0], [100, 0]) is None
assert incomes == original_incomes

print("All assertions passed.")

All assertions passed.


- The first assertion catches an incorrect saving-share formula or denominator.
- The second catches code that incorrectly removes negative saving.
- The fourth catches treating the exact €600-per-person eligibility boundary as ineligible.
- The fifth catches a transfer calculation that fails to apply the €100-per-person cap.